# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates step-by-step loading, exploration, and initial data processing for the FAIR^2 dataset using the `mlcroissant` library. All dataset entities (record sets, fields, columns) are referenced by their Croissant `@id` fields to ensure precision and reproducibility.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) which contains metadata and points to available data files.


In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

We will load dataset metadata and make the data ready for exploration using `mlcroissant`. The Croissant schema serves as a machine-readable manifest linking data files and describing all record sets by their `@id`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Croissant schema URL for dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset and print metadata summary
dataset = mlc.Dataset(croissant_url)
# For mlcroissant >=0.6.0 to_json() returns a dict.
metadata = dataset.metadata.to_json()
print(f"\033[1m{metadata['name']}\033[0m\n{metadata['description']}")


## 2. Data Overview

Next, we examine which record sets (tables) are defined in the Croissant schema. Each record set and field is identified by its `@id`, following best practices for reproducibility and programmatic access.

In [ ]:
# List all available record sets using their @id
print("Available record sets (by @id):")
if hasattr(dataset, 'record_sets'):
    record_sets = [r['@id'] for r in dataset.record_sets]
    pprint(record_sets)
else:
    # For current mlcroissant versions
    record_sets = [r['@id'] for r in dataset._json['recordSet']]
    pprint(record_sets)

# Let's print fields for each record set, referenced by their @id
for rs_id in record_sets:
    print(f"\nFields in record set '{rs_id}':")
    rs = dataset.get_record_set(rs_id)
    # Print the @id and name for each field in the record set
    for field in rs['field']:
        print(f"  {field['@id']}: {field.get('name','<no name>')}")

## 3. Data Extraction

We can now extract the data from each record set (referenced by its `@id`) to pandas DataFrames for further exploration. We'll demonstrate this by loading *all available record sets* into dataframes, keyed by their `@id`.

You can look up the `@id` of any field in the previous section to use for targeted analysis.

In [ ]:
# Extract all record sets to pandas DataFrames
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"  (No records found for {record_set_id})")
        continue
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"  Columns: {dataframes[record_set_id].columns.tolist()}")

# For demonstration, pick the first record set with data
if dataframes:
    main_rs_id = next(iter(dataframes.keys()))
    print(f"\nExample rows from first record set: {main_rs_id}")
    display(dataframes[main_rs_id].head())
else:
    print("No extractable data was found in available record sets.")

## 4. Exploratory Data Analysis (EDA)

The following section shows typical data wrangling tasks:
- Filtering records by a numeric field,
- Normalizing the numeric column,
- Grouping by a key attribute.

Remember, **all columns are referenced by their Croissant `@id`**. Please replace variable names below with actual field `@id`s observed above when applying to a specific record set.

In [ ]:
# Example EDA on main record set
import numpy as np

# Set these variable IDs (edit based on output above):
record_set_id = main_rs_id  # replace if analyzing a different record set

# For demonstration, try to pick the first numeric field from DataFrame
df = dataframes[record_set_id]

# Attempt to pick a likely numeric column by dtype
numeric_field_candidates = [c for c in df.columns if np.issubdtype(df[c].dropna().dtype, np.number)]
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]
    print(f"Numeric field chosen: {numeric_field_id}")

    # Example filter: values greater than threshold
    threshold = df[numeric_field_id].quantile(0.75)  # use 75th percentile as demo threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with field '{numeric_field_id}' > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by first non-numeric field
    non_numeric = [c for c in df.columns if not np.issubdtype(df[c].dropna().dtype, np.number)]
    if non_numeric:
        group_field_id = non_numeric[0]
        print(f"\nGrouping by '{group_field_id}':")
        grouped = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        display(grouped.head())
    else:
        print("No non-numeric column available for grouping.")
else:
    print("No numeric field available in this record set to perform EDA.")

## 5. Visualization

This section demonstrates how to visualize data from the primary record set. For example, plotting the distribution of a selected numeric field or relationship between two fields.

In [ ]:
import matplotlib.pyplot as plt

if 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of field: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Scatter with group (if available)
    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        for k, sub in df.groupby(group_field_id):
            plt.scatter(
                [k]*len(sub),
                sub[numeric_field_id],
                label=str(k), alpha=0.3
            )
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion

In this notebook, we loaded and explored the FAIR² dataset using the `mlcroissant` library, strictly referencing all entities by their Croissant `@id` for robust and unambiguous processing. We performed basic EDA steps and visualized the structure and data attributes as defined in the dataset schema. For advanced analyses or custom transformations, always refer to the field and record set `@id`s obtained programmatically for reproducibility.
